# 02 · Exploratory analysis

Questions this notebook answers before any modeling:
1. How much of Massachusetts gas demand is weather, and how much is structure?
2. Is demand trending down once weather is removed (efficiency, electrification)?
3. What moves Henry Hub, and how are New England demand and national price connected?

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, data, features, plotting
plotting.use_style()
pd.set_option("display.float_format", "{:,.2f}".format)
from src import demand_backtest as dbt
raw = data.load_raw()
dfd = features.build_demand_frame(raw["ma_consumption"], raw["weather_ma"])
cons = raw["ma_consumption"]

## 1 · Demand by end use

In [ ]:
annual = cons.drop(columns="total").resample("YS").sum() / 1000  # Bcf
annual = annual[annual.index.year < cons.index.max().year]  # complete years only
fig, ax = plt.subplots()
for c in annual.columns:
    ax.plot(annual.index.year, annual[c], color=plotting.SECTOR_COLORS[c], label=c.replace("_", " ").title())
ax.set_title("Massachusetts gas deliveries by end use")
ax.set_ylabel("Bcf per year")
ax.legend(ncol=4, loc="upper left")
plotting.source_note(ax, "Source: EIA Natural Gas Monthly")
plt.show()

## 2 · The heating curve
Monthly volume per day against HDD per day. The slope is the weather sensitivity; the intercept is base load (water heating, cooking, industry, power burn).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
recent = dfd[dfd.index.year >= dfd.index.year.max() - 5]
older = dfd[dfd.index.year < dfd.index.year.max() - 5]
ax.scatter(older["hdd_per_day"], older["y"] / older["days"], s=14, color=plotting.NEUTRAL, alpha=.5, label="Earlier years")
ax.scatter(recent["hdd_per_day"], recent["y"] / recent["days"], s=18, color=plotting.SERIES[0], label="Last 5 years")
for sub, col in ((older, plotting.NEUTRAL), (recent, plotting.SERIES[0])):
    b, a = np.polyfit(sub["hdd_per_day"], sub["y"] / sub["days"], 1)
    xs = np.linspace(0, dfd["hdd_per_day"].max(), 50)
    ax.plot(xs, a + b * xs, color=col, lw=1.5)
ax.set_xlabel("Heating degree days per day")
ax.set_ylabel("MMcf per day")
ax.set_title("Demand vs. weather")
ax.legend()
plt.show()
r = np.corrcoef(dfd["hdd"], dfd["y"])[0, 1]
print(f"Correlation of monthly HDD with monthly volume: r = {r:.3f} (R² = {r*r:.2f})")

## 3 · Weather-normalized demand
The standard regulatory adjustment: what each month would have used in normal weather. This separates a warm winter from a real decline in demand.

In [ ]:
wn = dbt.weather_normalize(dfd)
print(f"Weather sensitivity: {wn.attrs['mmcf_per_hdd']:.1f} MMcf per HDD")
roll = wn[["y", "y_wn"]].rolling(12).sum().dropna() / 1000
fig, ax = plt.subplots()
ax.plot(roll.index, roll["y"], color=plotting.NEUTRAL, lw=1.5, label="Actual (12-mo total)")
ax.plot(roll.index, roll["y_wn"], color=plotting.SERIES[0], label="Weather-normalized (12-mo total)")
ax.set_ylabel("Bcf")
ax.set_title("Is Massachusetts gas demand declining once weather is removed?")
ax.legend()
plt.show()
yrs = wn["y_wn"].resample("YS").sum()
yrs = yrs[(yrs.index.year >= yrs.index.year.max() - 10) & (yrs.index.year < yrs.index.year.max())]
slope = np.polyfit(yrs.index.year, yrs.values, 1)[0]
print(f"Trend in weather-normalized demand, last 10 complete years: {slope/yrs.mean()*100:+.2f}% per year")

## 4 · Henry Hub and storage

In [ ]:
dfp = features.build_price_frame(raw["henry_hub"], raw["storage_l48"], raw["lng_exports"], raw["weather_us"], risk=raw["risk"])
fig, ax = plt.subplots()
ax.plot(dfp.index, dfp["price"], color=plotting.SERIES[0], lw=1.2)
ax.set_title("Henry Hub spot price, weekly average")
ax.set_ylabel("$/MMBtu")
plotting.source_note(ax, "Source: EIA")
plt.show()

fig, ax = plt.subplots()
s = dfp["storage_surplus_pct"].dropna()
ax.fill_between(s.index, 0, s.where(s > 0), color=plotting.SERIES[0], alpha=.6, lw=0, label="Above 5-yr avg")
ax.fill_between(s.index, 0, s.where(s < 0), color=plotting.SERIES[1], alpha=.6, lw=0, label="Below 5-yr avg")
ax.axhline(0, color=plotting.INK_2, lw=.8)
ax.set_title("Lower 48 storage vs. prior 5-year average")
ax.set_ylabel("% difference")
ax.legend(loc="upper left")
plt.show()

Storage surplus is the classic fundamental: a surplus heading into winter usually pressures price. Does it show up in the *next* month's price change?

In [ ]:
d = dfp.dropna(subset=["storage_surplus_pct", "y_h4"])
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(d["storage_surplus_pct"], d["y_h4"] * 100, s=8, color=plotting.SERIES[0], alpha=.35)
ax.axhline(0, color=plotting.INK_2, lw=.8)
ax.set_xlabel("Storage vs. 5-yr avg (%), known at forecast time")
ax.set_ylabel("Price change next 4 weeks (%)")
ax.set_title("Storage surplus vs. next-month price change")
plt.show()
print(f"Correlation: {d['storage_surplus_pct'].corr(d['y_h4']):.3f}")

## Takeaways
* Weather explains the large majority of month-to-month demand variation, so the demand models should be built around HDD.
* Weather-normalized demand is the right lens for structural trends; that trend is what electrification and efficiency programs would move.
* Weekly price changes are noisy and only weakly linked to fundamentals, so the price model must be judged against a random walk, not against zero.